# Cube Nano - 95-Cloud training on Google Colab

Pipeline: Kaggle download -> paired preprocessing -> scene-level split -> validation -> RGB training -> evaluation.

Before running the Kaggle cell, enable a GPU in **Runtime > Change runtime type** and have a Kaggle API token ready.

In [ ]:
!pip -q install kaggle tifffile tqdm onnx
import os, json, shutil, subprocess, sys
from pathlib import Path
from google.colab import files

PROJECT = Path('/content/cube_nano')
RAW = Path('/content/95cloud_kaggle')
RUN = Path('/content/cloud_95cloud_run')
KAGGLE_SLUG = 'sorour/95cloud-cloud-segmentation-on-satellite-images'
assert shutil.which('kaggle'), 'Kaggle CLI was not installed'
print('Colab workspace ready:', RUN)

In [ ]:
# Upload kaggle.json from https://www.kaggle.com/settings
!pip -q install --upgrade kaggle

import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Nhập Kaggle API token: ")

RAW.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        "kaggle", "datasets", "download",
        "-d", KAGGLE_SLUG,
        "-p", str(RAW),
        "--unzip",
    ],
    check=True,
)

print("Downloaded files:", sum(1 for p in RAW.rglob("*") if p.is_file()))

In [ ]:
# Clone the current source so the notebook uses src/ rather than a duplicated training loop.
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/hoxuanphu/cube_nano.git', str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / 'src'))
from data.preprocess_95cloud import discover_scene_files

# Find the actual dataset root regardless of Kaggle's nested extraction layout.
tiff_parents = {p.parent for p in RAW.rglob('*.TIF')} | {p.parent for p in RAW.rglob('*.tif')}
candidates = []
for parent in tiff_parents:
    candidates.extend([parent, *parent.parents[:5]])
candidates = list(dict.fromkeys(sorted(candidates, key=lambda p: (len(p.parts), str(p)))))
DATA_ROOT = None
for candidate in [RAW, *candidates]:
    try:
        if len(discover_scene_files(candidate, channels=3)) > 0:
            DATA_ROOT = candidate
            break
    except (FileNotFoundError, ValueError):
        pass
if DATA_ROOT is None:
    raise FileNotFoundError(f'Could not locate complete 95-Cloud scenes under {RAW}')
print('Dataset root:', DATA_ROOT)
print('Scenes:', len(discover_scene_files(DATA_ROOT, channels=3)))

In [ ]:
RUN.mkdir(parents=True, exist_ok=True)
def run(script, *args):
    command = [sys.executable, str(PROJECT / script), *map(str, args)]
    print('$', ' '.join(command))
    subprocess.run(command, cwd=RUN, check=True)

run('src/data/preprocess_95cloud.py', '--data_dir', DATA_ROOT, '--out_dir', RUN/'data/processed/all', '--patch_size', 384, '--cloud_ratio_threshold', 0.10, '--channels', 3, '--force')
run('src/data/split_dataset.py', '--src_dir', RUN/'data/processed/all', '--out_dir', RUN/'data/processed', '--val_ratio', 0.15, '--test_ratio', 0.15, '--seed', 42, '--force')
print('Preprocessing and scene split completed.')

In [ ]:
# Validate one batch before spending GPU time on training.
import torch
from torch.utils.data import DataLoader
from data.cloud_dataset import CloudDataset
train_ds = CloudDataset(RUN/'data/processed/train', is_train=True, target_channels=3, channel_dropout_p=0, crop_size=256, cloud_ratio_threshold=0.10)
x, y = next(iter(DataLoader(train_ds, batch_size=4, num_workers=0)))
assert x.shape == (4, 3, 256, 256) and x.dtype == torch.float32
assert torch.isfinite(x).all() and torch.all((y == 0) | (y == 1))
print('Smoke check:', tuple(x.shape), x.dtype, 'labels:', y.tolist(), 'cuda:', torch.cuda.is_available())

In [ ]:
run('src/train.py', '--train_dir', RUN/'data/processed/train', '--val_dir', RUN/'data/processed/val', '--out_dir', RUN/'checkpoints', '--epochs', 12, '--batch_size', 32, '--lr', 1e-3, '--channels', 3, '--crop_size', 256, '--cloud_ratio_threshold', 0.10, '--channel_dropout_p', 0, '--num_workers', 2, '--seed', 42, '--threshold', 0.5)
run('src/eval.py', '--test_dir', RUN/'data/processed/test', '--model_path', RUN/'checkpoints/best_model.pth', '--batch_size', 32, '--channels', 3, '--crop_size', 256, '--cloud_ratio_threshold', 0.10, '--num_workers', 2, '--seed', 42, '--threshold', 0.5)
print('Artifacts:', list((RUN/'checkpoints').glob('*')), list((RUN/'results').glob('*')))

In [ ]:
# Download results to your computer before the Colab runtime expires.
shutil.make_archive('/content/cloud_95cloud_results', 'zip', RUN)
files.download('/content/cloud_95cloud_results.zip')